# Read a flotation bank as a kinetic response

**Question:** Why does shortening a continuous bank change recovery differently from extending a batch test?

Level: 1. Prerequisites: None; begin here. Reading time: 18 minutes.



## Objectives

Run the Cu-Mo rougher and distinguish its virtual batch curve from the continuously fed bank response. Understand why recovery versus time is a process response, not a decorative animation.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

For the level 1 02-flotation lesson, record the pinned versions before analysis so the environment behind its tables and figures can be reconstructed.

In [ ]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Data

Owner: OreFlow authors, MIT project licence. The authored copper-molybdenum circuit is supplied by pipeline.cases.catalog. The production engine calculates rougher feed, class rates and residence; batch sample times and numerical fitting settings are read through engine/data/constants.json. Nominal, shortened-bank and changed-dose records are freshly generated where the lesson uses them. Virtual batch values and the exact-rate reference share simulator assumptions and are not laboratory measurements. Source: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

OreFlow aggregates class-specific rates as $R_b(t)=\sum_j c_jx_j(1-e^{-k_jt})/\sum_j c_jx_j$. A continuous bank has a different residence-time distribution and is calculated separately. Call `simulate` and the production `trace` builder.

Select one authored Cu-Mo case and run the same circuit and trace methods used by OreFlow's data pipeline.

In [ ]:
# Solve the rougher and serialize its kinetic record with the production trace builder.
from pipeline.cases.catalog import CASES
from pipeline.engine.circuit import simulate
from pipeline.engine.trace import trace

case = next((item for item in CASES if item.id == "copper_molybdenum"))
result = simulate(case.ore, case.plant, case.nominal)
record = trace(result, case.nominal, case.plant.family)
kinetics = record["methods"]["kinetics"]
print("family:", record["family"], "| status:", kinetics["status"])
print("bank residence (min):", round(kinetics["bank"]["residence_min"], 2))
print("engine rougher recovery (%):", round(kinetics["bank"]["engine_rougher_pct"], 2))


Plot batch recovery against time and continuous-bank recovery against cell position; the x-axes represent different residence models.

In [ ]:
# Plot batch recovery against time and continuous-bank recovery against cell position; the x-axes
# represent different residence models.
plt.rcParams.update({"font.size": 22, "axes.titlepad": 8})
bank = record["curves"]["bank_profile"]
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8))
axes[0].plot(kinetics["times_min"], kinetics["batch_recovery_pct"], "o-", label="Virtual batch")
axes[0].set(xlabel="Batch time (min)", ylabel="Recovery\n(%)", title="Virtual batch test")
axes[1].plot([v["cell"] for v in bank], [100 * v["recovery"] for v in bank], "o-", color="#b34d32")
axes[1].set(xlabel="Bank cell number", ylabel="Recovery\n(%)", title="Continuous-bank profile")
for ax in axes:
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.25)
fig.subplots_adjust(left=0.24, right=0.96, bottom=0.15, top=0.91, hspace=1.0)


Batch time and continuous-cell position are different exposure coordinates. The batch curve lets every particle experience the stated duration, whereas a bank mixes residence times and transfers material between stages. Consequently similar-looking curves do not justify equating their x axes. The following stream table tracks valuable grade and solids so a recovery change can be distinguished from simple dilution or extra gangue collection.

Read rougher feed, concentrate and tailings together. Grade and recovery answer different questions.

In [ ]:
# Read rougher feed, concentrate and tailings together.
streams = record["streams"]
rows = [
    (name, streams[name]["solids_tph"], streams[name]["grades"][kinetics["species"]])
    for name in ("rougher_feed", "rougher_concentrate", "rougher_tail")
    if name in streams
]
display(pd.DataFrame(rows, columns=["Stream", "Solids (t/h)", f"{kinetics['species']} grade (%)"]))


Return to the copper-molybdenum rougher and generate a fresh kinetic record from its particle-class rates. Five named forms are fitted to the same virtual batch observations. The recorded continuous-bank recovery uses a residence model rather than the batch's single exposure time. This fresh fit connects the two views above without treating either curve as laboratory data. Notice that true flotation and overall rougher recovery need not coincide because entrainment contributes to the latter; the exact kinetic reference intentionally isolates true flotation.

In [ ]:
# Recalculate the nominal copper-molybdenum rougher and fit its virtual batch response with the
# production kinetic routine.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.kinetics import kinetic_record

case = CASE_BY_ID["copper_molybdenum"]
nominal = simulate(case.ore, case.plant, case.nominal)
fits = kinetic_record(
    nominal.flotation,
    nominal.ore,
    case.nominal.rougher_cells,
    nominal.metrics["rougher_recovery_pct"],
)
display(
    pd.DataFrame(
        [
            {
                "Model": m["id"],
                "Batch RMSE (pp)": m["rmse_pct"],
                "Bank error (pp)": m["lumping_error_pct"],
            }
            for m in fits["models"]
        ]
    )
)


Remove one continuous cell, leaving feed and collector at their nominal settings. There is now less installed tank volume and less total residence time for that flow. The engine calculates the shortened bank instead of truncating the batch curve. Inspect its recovery before fitting any approximation. This order matters because a fitted equation cannot supply an independent physical reference for its own projection. The calculation remains local to this authored bank and does not imply that one cell always has the same recovery contribution elsewhere.

In [ ]:
# Reduce the number of continuous cells by one without changing the feed or reagent dose.
short_point = case.nominal.with_values(rougher_cells=max(1, case.nominal.rougher_cells - 1))
short = simulate(case.ore, case.plant, short_point)
print("Short-bank recovery (%):", round(short.metrics["rougher_recovery_pct"], 3))


Fit the shortened-bank kinetic record and print its exact true-flotation reference. A particle with a small rate may remain poorly recovered over the available bank residence even when a long batch test would approach a high ultimate recovery. This explains why extending batch time and adding a continuous cell are different interventions. Compare the two exact references and their fitted forms while preserving the distinction between valuable recovery, solids recovery and concentrate grade. The lesson teaches residence modeling, not plant-scale tank sizing or a recommended cell count.

In [ ]:
# Fit and project the same kinetic family after shortening the bank.
short_fits = kinetic_record(
    short.flotation, short.ore, short_point.rougher_cells, short.metrics["rougher_recovery_pct"]
)
print("Exact short-bank recovery (%):", round(short_fits["bank"]["exact_true_flotation_pct"], 3))


The exact bank reference excludes mechanical entrainment, while the stream recovery includes every route by which valuable mass reaches concentrate. Those quantities can differ without an arithmetic error. To diagnose a discrepancy, identify the payable species, the residence basis and whether water carries fines into concentrate. A batch kinetic equation models collection exposure; it cannot by itself reconstruct all water and solids transfers in the circuit. That is why this lesson keeps the process solve alongside its kinetic summary.

## Results

**Answer:** Removing one cell leaves overall rougher recovery <!-- result:Short-bank overall recovery (%) -->0<!-- /result --> percent and a true-flotation reference of <!-- result:Short-bank true recovery (%) -->0<!-- /result --> percent. Batch exposure and continuous-bank residence therefore remain distinct calculations; entrainment also separates overall recovery from the kinetic reference.

Collect short-bank overall/true recovery alongside nominal kinetic errors for the residence interpretation.

In [ ]:
# Summarize the computed outcomes used in this lesson's interpretation.
kinetic_by_name = {model["id"]: model for model in fits["models"]}
answer_values = {
    "Short-bank overall recovery (%)": short.metrics["rougher_recovery_pct"],
    "Short-bank true recovery (%)": short_fits["bank"]["exact_true_flotation_pct"],
    "First-order batch RMSE (pp)": kinetic_by_name["first_order"]["rmse_pct"],
    "Kelsall batch RMSE (pp)": kinetic_by_name["kelsall"]["rmse_pct"],
    "First-order bank error (pp)": abs(kinetic_by_name["first_order"]["lumping_error_pct"]),
    "Kelsall bank error (pp)": abs(kinetic_by_name["kelsall"]["lumping_error_pct"]),
}
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Summary

The lesson executed OreFlow's solver and inspected its simulated batch and bank responses. The record is not laboratory data, and a visual step through the circuit is not physical plant time.

## References

- https://doi.org/10.1016/0301-7516(81)90019-3
- https://doi.org/10.1016/0301-7516(81)90019-3
- https://doi.org/10.1016/0301-7516(81)90019-3